# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [4]:
# Load the libraries as required.
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder, MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
import shap
import numpy as np
import pickle

/opt/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [11]:
# Splitting features and target variable
X = fires_dt.drop(columns=['area'])
y = fires_dt['area']

In [12]:
# Splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [34]:
preproc1 = ColumnTransformer(transformers=[
    ('num', StandardScaler(), ['ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']),
    ('cat', OneHotEncoder(), ['month', 'day'])
])

In [14]:
# Identify numerical and categorical features
num_features = ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']
cat_features = ['month', 'day']

# Preprocessing Pipelines
preproc1 = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [35]:
preproc2 = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('scaler', StandardScaler()),
        ('transform', PowerTransformer(method='yeo-johnson'))
    ]), ['ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']),
    ('cat', OneHotEncoder(), ['month', 'day'])
])


In [16]:
preproc2 = ColumnTransformer([
    ('num', Pipeline([
        ('scaler', StandardScaler()),
        ('transform', PowerTransformer())
    ]), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [19]:
# Define baseline regressor (KNeighborsRegressor)
baseline_regressor = KNeighborsRegressor()

In [20]:
# Define advanced regressor (RandomForestRegressor)
advanced_regressor = RandomForestRegressor()

In [21]:
# Define parameter grids for grid search
param_grid_baseline = {
    'regressor__n_neighbors': [3, 5, 7, 9]  # Hyperparameter to tune for baseline regressor
}

In [22]:
param_grid_advanced = {
    'regressor__n_estimators': [50, 100, 150],  # Hyperparameter to tune for advanced regressor
    'regressor__max_depth': [None, 10, 20]  # Another hyperparameter to tune for advanced regressor
}

In [23]:

# Pipeline A = preproc1 + baseline
pipeline_A = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', baseline_regressor)
])

In [24]:
# Pipeline B = preproc2 + baseline
pipeline_B = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', baseline_regressor)
])

In [25]:
# Pipeline C = preproc1 + advanced model
pipeline_C = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', advanced_regressor)
])

In [26]:
# Pipeline D = preproc2 + advanced model
pipeline_D = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', advanced_regressor)
])
    

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [36]:
param_grid = {'regressor__alpha': [0.1, 1, 10]}
grid_A = GridSearchCV(pipeline_A, param_grid, scoring='neg_mean_squared_error', cv=5)
grid_B = GridSearchCV(pipeline_B, param_grid, scoring='neg_mean_squared_error', cv=5)

param_grid_rf = {'regressor__n_estimators': [50, 100, 200], 'regressor__max_depth': [None, 10, 20]}
grid_C = GridSearchCV(pipeline_C, param_grid_rf, scoring='neg_mean_squared_error', cv=5)
grid_D = GridSearchCV(pipeline_D, param_grid_rf, scoring='neg_mean_squared_error', cv=5)


In [27]:
# Define a function to perform grid search and cross-validation
def evaluate_pipeline(pipeline, param_grid):
    grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)
    cv_scores = cross_val_score(grid_search.best_estimator_, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    return grid_search.best_estimator_, np.sqrt(-cv_scores.mean())

# Evaluate

+ Which model has the best performance?

In [38]:
# Evaluate each pipeline
best_models = {}
for name, pipeline in {'Pipeline A': pipeline_A, 'Pipeline B': pipeline_B, 'Pipeline C': pipeline_C, 'Pipeline D': pipeline_D}.items():
    if 'A' in name or 'B' in name:
        best_model, rmse = evaluate_pipeline(pipeline, param_grid_baseline)
    else:
        best_model, rmse = evaluate_pipeline(pipeline, param_grid_advanced)
    best_models[name] = (best_model.named_steps['regressor'], rmse)  # Extracting the regressor from the pipeline

# Find the best performing model
best_model_name, (best_model, best_rmse) = min(best_models.items(), key=lambda x: x[1][1])

# Save the best performing model
with open("best_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Best performing model:", best_model_name)
print("RMSE:", best_rmse)

Best performing model: Pipeline A
RMSE: 47.4762891960525


# Export

+ Save the best performing model to a pickle file.

In [39]:
import pickle

# Save the best performing model to a pickle file
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_models['Pipeline B'][0], f)


In [40]:
pip install shap

Note: you may need to restart the kernel to use updated packages.


# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [46]:
import shap
import numpy as np
from sklearn.preprocessing import OneHotEncoder



In [ ]:

# Load the best performing model from the saved pickle file
with open("best_model.pkl", "rb") as f:
    best_model = pickle.load(f)

# Preprocess the data to ensure all features are numeric
# Assuming X_train and X_test are your original dataframes
# Perform one-hot encoding for categorical features
categorical_features = X_train.select_dtypes(include=['object']).columns
encoder = OneHotEncoder(drop='first').fit(X_train[categorical_features])
X_train_encoded = encoder.transform(X_train[categorical_features])
X_test_encoded = encoder.transform(X_test[categorical_features])

# Concatenate encoded features with the original numeric features
X_train_processed = np.hstack((X_train_encoded.toarray(), X_train.select_dtypes(exclude=['object'])))
X_test_processed = np.hstack((X_test_encoded.toarray(), X_test.select_dtypes(exclude=['object'])))

# Create a SHAP KernelExplainer for the best-performing model
explainer = shap.KernelExplainer(best_model.predict, X_train_processed)

# Select an observation from the test set for explanation
observation_index = 0  # Change this index as needed
observation = X_test_processed[observation_index:observation_index+1]

# Explain the prediction for the selected observation
shap_values = explainer.shap_values(observation)

# Plot the SHAP values for the selected observation
shap.summary_plot(shap_values, features=observation, feature_names=X.columns)

# Get the SHAP values for all observations in the training set
shap_values_train = explainer.shap_values(X_train_processed)

# Get the mean absolute SHAP values for each feature across the training set
mean_abs_shap_values = np.mean(np.abs(shap_values_train), axis=0)

# Sort the features based on their importance (absolute SHAP values)
sorted_indices = np.argsort(mean_abs_shap_values)[::-1]
sorted_features = X.columns[sorted_indices]

# Print the most and least important features
print("Most important features:")
for feature in sorted_features[:5]:
    print(feature)
print("\nLeast important features:")
for feature in sorted_features[-5:]:
    print(feature)

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.